In [ ]:
import os

import random
import json
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, Dropout, BatchNormalization, Concatenate, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.config.experimental.enable_op_determinism()

In [ ]:
# === Load and filter stock data ===
stocks = ['ENB','GS','WFC','GME','D','EA','CMCSA','DHI','CRM','VRTX',
          'SPWR','GILD','WDC','BX','AAL']

df = (
    pd.read_csv('fnspid_prices_title_sentiment.csv',
                parse_dates=['date'],
                index_col='date')
      .query("Stock_symbol in @stocks")
)

# === Generate binary target ===
def make_target(group):
    group['target_binary'] = (group['adj close'].shift(-1) > group['adj close']).astype(int)
    return group

df = (
    df.groupby('Stock_symbol', group_keys=False)
      .apply(make_target)
      .dropna(subset=['target_binary'])
)

# === Helper to load and daily-resample macro data ===
def load_macro(path, date_col='observation_date'):
    return (
        pd.read_csv(path, parse_dates=[date_col], index_col=date_col)
          .resample('D')
          .fillna(method='ffill').fillna(method='bfill')
    )

# === Load and combine all macro series ===
macro_paths = {
    'DFF': 'interest_rates.csv',
    'CPIAUCSL': 'cpi.csv',
    'UNRATE': 'unemployment_rate.csv',
    'PPI': 'ppi.csv',
    'GOLD_OIL_RATES': 'gold_silver_rates_oil_1999_2024.csv',
    'DTWEXBGS': 'nominal_us_dollar_index.csv'
}
macro_dfs = [load_macro(p) for p in macro_paths.values()]
merged_macro = (
    pd.concat(macro_dfs, axis=1)
      .reindex(df.index.unique())  # match only stock dates
      .fillna(method='ffill')      # forward-fill missing macro values
      .fillna(method='bfill')      # backfill start gaps if any
)

# === Join stock data with macro data ===
merged = df.join(merged_macro, how='left')
merged = merged.drop(columns=['Vol._x', 'Vol._y'])
# === Check ===
print(merged.info())
print(merged.head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 33660 entries, 2015-01-05 to 2023-12-01
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   volume                           33660 non-null  float64
 1   open                             33660 non-null  float64
 2   high                             33660 non-null  float64
 3   low                              33660 non-null  float64
 4   close                            33660 non-null  float64
 5   adj close                        33660 non-null  float64
 6   Stock_symbol                     33660 non-null  object 
 7   avg_weighted_sent                33660 non-null  float64
 8   avg_score                        33660 non-null  float64
 9   article_count                    33660 non-null  float64
 10  movement_percent                 33660 non-null  float64
 11  target_binary                    33660 non-null  int64  
 12  D

/tmp/ipython-input-634012813.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_target)
/tmp/ipython-input-634012813.py:28: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use obj.ffill(), obj.bfill(), or obj.nearest() instead.
  .fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-634012813.py:28: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-634012813.py:28: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use o

In [ ]:
# ======================
# Reproducibility
# ======================
def set_seed(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ======================
# Sequence generator
# ======================
def make_sequences_dual(df, window_size, price_cols, sent_cols, target, skip_all_zero_sent=True):
    X_price, X_sent, y = [], [], []
    for i in range(len(df) - window_size):
        price_seq = df.iloc[i:i+window_size][price_cols].values
        sent_seq = df.iloc[i:i+window_size][sent_cols].values
        if skip_all_zero_sent and np.all(sent_seq == 0):
            continue
        target_val = df.iloc[i+window_size][target]
        X_price.append(price_seq)
        X_sent.append(sent_seq)
        y.append(target_val)
    return np.array(X_price), np.array(X_sent), np.array(y)

# ======================
# Temporal Attention Layer
# ======================
class TemporalAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, x):
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        ait = tf.nn.softmax(tf.tensordot(uit, self.u, axes=1), axis=1)
        out = tf.reduce_sum(x * ait, axis=1)
        return out

# ======================
# Feature Attention Layer
# ======================
class FeatureAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform',
                                 trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform',
                                 trainable=True)
        super().build(input_shape)

    def call(self, x):
        # x: (batch, timesteps, features)
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)  # (batch, timesteps, features)
        ait = tf.nn.softmax(tf.tensordot(uit, self.u, axes=1), axis=-1)  # attention over features
        out = x * ait
        return out

# ======================
# Model builder with feature attention
# ======================
def build_model_from_config(window_size, price_n_features, sent_n_features, config):
    l2_value = config.get('l2_value', 1e-4)
    learning_rate = config.get('learning_rate', 1e-4)

    # Price branch
    price_in = Input(shape=(window_size, price_n_features), name='price_input')
    x = price_in

    # Apply CNN / BatchNorm / Dropout layers before LSTM
    for layer_cfg in config.get('price_stack', []):
        t = layer_cfg['type'].lower()
        if t == 'conv1d':
            x = Conv1D(filters=layer_cfg.get('filters', 16),
                       kernel_size=layer_cfg.get('kernel_size', 3),
                       activation=layer_cfg.get('activation', 'relu'),
                       padding=layer_cfg.get('padding', 'same'))(x)
        elif t == 'batchnorm':
            x = BatchNormalization()(x)
        elif t == 'dropout':
            x = Dropout(rate=layer_cfg.get('rate', 0.2))(x)
        elif t == 'lstm':
            break  # stop before LSTM

    # Feature attention before LSTM if enabled
    if config.get('feature_attention', False):
        x = FeatureAttention()(x)

    # Apply LSTM layers from price_stack
    lstm_layers = [l for l in config.get('price_stack', []) if l['type'].lower() == 'lstm']
    for i, layer_cfg in enumerate(lstm_layers):
        # Return sequences True for all except last
        return_seq = layer_cfg.get('return_sequences', True)
        # if i == len(lstm_layers) - 1:
        #     return_seq = False  # final LSTM outputs (batch, features) for merging
        x = LSTM(layer_cfg.get('units', 32),
                 recurrent_dropout=layer_cfg.get('recurrent_dropout', 0.0),
                 return_sequences=return_seq)(x)

    # Apply temporal attention to sent branch
    if config.get('price_attention', True):
        x = TemporalAttention()(x)

    # Sentiment branch
    sent_in = Input(shape=(window_size, sent_n_features), name='sent_input')
    s = sent_in
    for layer_cfg in config.get('sent_stack', []):
        t = layer_cfg['type'].lower()
        if t == 'lstm':
            s = LSTM(layer_cfg.get('units', 8),
                     recurrent_dropout=layer_cfg.get('recurrent_dropout', 0.0),
                     return_sequences=layer_cfg.get('return_sequences', True))(s)
        elif t == 'dropout':
            s = Dropout(rate=layer_cfg.get('rate', 0.2))(s)
        elif t == 'batchnorm':
            s = BatchNormalization()(s)

    # Apply temporal attention to sent branch
    if config.get('sent_attention', True):
        s = TemporalAttention()(s)
    else:
        s = LSTM(config.get('sent_final_lstm_units', 8), return_sequences=False)(s)

    # Merge and output
    merged = Concatenate()([x, s])
    out = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_value))(merged)

    model = Model(inputs=[price_in, sent_in], outputs=out)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['AUC', 'accuracy'])
    return model



In [ ]:
def overlapping_fixed_tscv(n_samples, n_splits, train_size, val_size, test_size):
    """
    Fixed-size overlapping rolling time series splits.
    Each fold shifts forward by step = test_size.
    """
    total_window = train_size + val_size + test_size
    step = test_size  # overlap controlled by test window

    for i in range(n_splits):
        start = i * step
        end = start + total_window
        if end > n_samples:
            break
        train_idx = np.arange(start, start + train_size)
        val_idx = np.arange(start + train_size, start + train_size + val_size)
        test_idx = np.arange(start + train_size + val_size, start + total_window)
        yield train_idx, val_idx, test_idx

# ======================
# Training loop (fixed)
# ======================
def train_on_merged(merged, config):
    set_seed(config.get('seed', 0))
    window_size = config['window_size']
    price_cols = config['price_cols']
    macro_cols = config['macro_cols']
    sent_cols = config['sent_cols']
    target = config['target']
    feature_cols = config['feature_cols'] or [f"{c}_logret" for c in price_cols] + [f"{c}_ret" for c in macro_cols]

    results = []

    for symbol in merged['Stock_symbol'].unique():
        print(f"\n==== Training for: {symbol} ====")
        df_symbol = merged[merged['Stock_symbol'] == symbol].copy()

        # Compute returns
        for col in price_cols:
            safe = df_symbol[col].replace(0, np.nan)
            df_symbol[f'{col}_logret'] = np.log(safe) - np.log(safe.shift(1))
        for col in macro_cols:
            df_symbol[f'{col}_ret'] = df_symbol[col].pct_change()
        df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')

        Xp_full, Xs_full, y_full = make_sequences_dual(df_symbol, window_size, feature_cols, sent_cols, target)
        if len(Xp_full) == 0:
            print(f"No sequences for {symbol}, skipping.")
            continue

        X_idx = np.arange(len(Xp_full))

        fold = 0
        for train_idx, val_idx, test_idx in overlapping_fixed_tscv(
                n_samples=len(X_idx),
                n_splits=5,
                train_size=1000,
                val_size=150,
                test_size=150):

            fold += 1
            print(f"\n-- Fold {fold} --")
            Xp_train, Xp_val, Xp_test = Xp_full[train_idx], Xp_full[val_idx], Xp_full[test_idx]
            Xs_train, Xs_val, Xs_test = Xs_full[train_idx], Xs_full[val_idx], Xs_full[test_idx]
            y_train, y_val, y_test = y_full[train_idx], y_full[val_idx], y_full[test_idx]

            def dist(name, y):
                ones = np.sum(y == 1)
                zeros = np.sum(y == 0)
                print(f"{name} -> 1: {ones/len(y)*100:.2f}% | 0: {zeros/len(y)*100:.2f}% (n={len(y)})")

            dist("Train", y_train)
            dist("Validation", y_val)
            dist("Test", y_test)

            model = build_model_from_config(window_size, len(feature_cols), len(sent_cols), config)
            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=config['early_stopping_patience'],
                restore_best_weights=True
            )

            model.fit(
                [Xp_train, Xs_train], y_train,
                validation_data=([Xp_val, Xs_val], y_val),
                epochs=config['epochs'],
                batch_size=config['batch_size'],
                callbacks=[early_stop],
                verbose=0
            )

            y_prob = model.predict([Xp_test, Xs_test])
            y_pred = (y_prob > 0.5).astype(int)

            auc = roc_auc_score(y_test, y_prob)
            f1 = f1_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            acc = np.mean(y_pred.flatten() == y_test.flatten())
            cm = confusion_matrix(y_test, y_pred)

            print(f"[{symbol}][Fold {fold}] AUC: {auc:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | ACC: {acc:.4f}")
            print(f"Confusion matrix:\n{cm}\n")

            results.append({
                'Symbol': symbol,
                'Fold': fold,
                'Test_AUC': auc,
                'Test_F1': f1,
                'Test_MCC': mcc,
                'Test_ACC': acc
            })

    results_df = pd.DataFrame(results)
    return results_df

In [ ]:
def summarize_results(results_df):
    """Summarize per-stock and overall mean AUC and ACC."""
    grouped = results_df.groupby('Symbol')

    acc_by_symbol = grouped['Test_ACC'].mean().sort_values(ascending=False)
    auc_by_symbol = grouped['Test_AUC'].mean().sort_values(ascending=False)
    mcc_by_symbol = grouped['Test_MCC'].mean().sort_values(ascending=False)
    f1_by_symbol = grouped['Test_F1'].mean().sort_values(ascending=False)


    print("=== Per-Stock Summary ===")
    print("\nMean Test Accuracy by Symbol:")
    print(acc_by_symbol)

    print("\nMean Test AUC by Symbol:")
    print(auc_by_symbol)

    print("\nMean Test MCC by Symbol:")
    print(mcc_by_symbol)

    print("\nMean Test F1 by Symbol:")
    print(f1_by_symbol)

    print("\n=== Overall Summary ===")
    print(f"Overall Mean ACC: {acc_by_symbol.mean():.4f}")
    print(f"Overall Mean AUC: {auc_by_symbol.mean():.4f}")
    print(f"Overall Mean MCC: {mcc_by_symbol.mean():.4f}")
    print(f"Overall Mean F1: {f1_by_symbol.mean():.4f}")

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 20,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}

# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



==== Training for: AAL ====

-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 123ms/step
[AAL][Fold 1] AUC: 0.4735 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5200
Confusion matrix:
[[78  0]
 [72  0]]


-- Fold 2 --
Train -> 1: 49.80% | 0: 50.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step
[AAL][Fold 2] AUC: 0.4962 | F1: 0.0563 | MCC: 0.0612 | ACC: 0.5533
Confusion matrix:
[[81  1]
 [66  2]]


-- Fold 3 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 1s 447ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
[AAL][Fold 3] AUC: 0.4888 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[76  0]
 [74  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 123ms/step
[AAL][Fold 4] AUC: 0.4418 | F1: 0.4487 | MCC: -0.1387 | ACC: 0.4267
Confusion matrix:
[[29 51]
 [35 35]]


-- Fold 5 --
Train -> 1: 47.50% | 0: 52.50% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step
[AAL][Fold 5] AUC: 0.5488 | F1: 0.1446 | MCC: -0.0202 | ACC: 0.5267
Confusion matrix:
[[73  8]
 [63  6]]


==== Training for: BX ====


/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 144ms/step
[VRTX][Fold 1] AUC: 0.5352 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 2 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step
[VRTX][Fold 2] AUC: 0.4609 | F1: 0.4118 | MCC: -0.0653 | ACC: 0.4667
Confusion matrix:
[[42 32]
 [48 28]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step
[VRTX][Fold 3] AUC: 0.5153 | F1: 0.6607 | MCC: 0.0000 | ACC: 0.4933
Confusion matrix:
[[ 0 76]
 [ 0 74]]


-- Fold 4 --
Train -> 1: 51.90% | 0: 48.10% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.40% | 0: 49.60% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step
[WDC][Fold 1] AUC: 0.4733 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4800
Confusion matrix:
[[72  0]
 [78  0]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 157ms/step
[WDC][Fold 2] AUC: 0.5268 | F1: 0.3200 | MCC: 0.0668 | ACC: 0.5467
Confusion matrix:
[[66 14]
 [54 16]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step
[WDC][Fold 3] AUC: 0.5598 | F1: 0.7124 | MCC: 0.0000 | ACC: 0.5533
Confusion matrix:
[[ 0 67]
 [ 0 83]]


-- Fold 4 --
Train -> 1: 51.20% | 0: 48.80% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.00% | 0: 50.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step
[WFC][Fold 1] AUC: 0.4991 | F1: 0.0000 | MCC: -0.1147 | ACC: 0.4933
Confusion matrix:
[[74  2]
 [74  0]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 219ms/step
[WFC][Fold 2] AUC: 0.5035 | F1: 0.3860 | MCC: 0.0486 | ACC: 0.5333
Confusion matrix:
[[58 21]
 [49 22]]


-- Fold 3 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 56.00% | 0: 44.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step
[WFC][Fold 3] AUC: 0.4867 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4400
Confusion matrix:
[[66  0]
 [84  0]]


-- Fold 4 --
Train -> 1: 48.60% | 0: 51.40% (n=1000)
Validation -> 1: 56.00% | 0: 44.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 20,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 196ms/step
[AAL][Fold 1] AUC: 0.5776 | F1: 0.6346 | MCC: 0.0330 | ACC: 0.4933
Confusion matrix:
[[ 8 70]
 [ 6 66]]


-- Fold 2 --
Train -> 1: 49.80% | 0: 50.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 203ms/step
[AAL][Fold 2] AUC: 0.4127 | F1: 0.2679 | MCC: -0.1455 | ACC: 0.4533
Confusion matrix:
[[53 29]
 [53 15]]


-- Fold 3 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step
[AAL][Fold 3] AUC: 0.4954 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[76  0]
 [74  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 58.67% | 0: 41.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 198ms/step
[BX][Fold 1] AUC: 0.4606 | F1: 0.6064 | MCC: -0.0479 | ACC: 0.5067
Confusion matrix:
[[19 43]
 [31 57]]


-- Fold 2 --
Train -> 1: 50.30% | 0: 49.70% (n=1000)
Validation -> 1: 58.67% | 0: 41.33% (n=150)
Test -> 1: 56.67% | 0: 43.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 208ms/step
[BX][Fold 2] AUC: 0.4561 | F1: 0.7117 | MCC: 0.0654 | ACC: 0.5733
Confusion matrix:
[[ 7 58]
 [ 6 79]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 56.67% | 0: 43.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 237ms/step
[BX][Fold 3] AUC: 0.5704 | F1: 0.7000 | MCC: 0.0569 | ACC: 0.5600
Confusion matrix:
[[ 7 60]
 [ 6 77]]


-- Fold 4 --
Train -> 1: 53.60% | 0: 46.40% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2

...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 194ms/step
[VRTX][Fold 1] AUC: 0.5541 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 2 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 179ms/step
[VRTX][Fold 2] AUC: 0.4552 | F1: 0.5556 | MCC: -0.0779 | ACC: 0.4667
Confusion matrix:
[[20 54]
 [26 50]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 199ms/step
[VRTX][Fold 3] AUC: 0.5132 | F1: 0.6516 | MCC: -0.0495 | ACC: 0.4867
Confusion matrix:
[[ 1 75]
 [ 2 72]]


-- Fold 4 --
Train -> 1: 51.90% | 0: 48.10% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.40% | 0: 49.60% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 206ms/step
[WDC][Fold 1] AUC: 0.5132 | F1: 0.0238 | MCC: -0.1444 | ACC: 0.4533
Confusion matrix:
[[67  5]
 [77  1]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step
[WDC][Fold 2] AUC: 0.5350 | F1: 0.5070 | MCC: 0.0642 | ACC: 0.5333
Confusion matrix:
[[44 36]
 [34 36]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 212ms/step
[WDC][Fold 3] AUC: 0.5589 | F1: 0.7124 | MCC: 0.0000 | ACC: 0.5533
Confusion matrix:
[[ 0 67]
 [ 0 83]]


-- Fold 4 --
Train -> 1: 51.20% | 0: 48.80% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.00% | 0: 50.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 196ms/step
[WFC][Fold 1] AUC: 0.5213 | F1: 0.0500 | MCC: -0.0653 | ACC: 0.4933
Confusion matrix:
[[72  4]
 [72  2]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 187ms/step
[WFC][Fold 2] AUC: 0.5103 | F1: 0.3137 | MCC: 0.0437 | ACC: 0.5333
Confusion matrix:
[[64 15]
 [55 16]]


-- Fold 3 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 56.00% | 0: 44.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step
[WFC][Fold 3] AUC: 0.4455 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4400
Confusion matrix:
[[66  0]
 [84  0]]


-- Fold 4 --
Train -> 1: 48.60% | 0: 51.40% (n=1000)
Validation -> 1: 56.00% | 0: 44.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 20,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}

# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 259ms/step
[AAL][Fold 1] AUC: 0.5828 | F1: 0.5802 | MCC: 0.1035 | ACC: 0.5467
Confusion matrix:
[[35 43]
 [25 47]]


-- Fold 2 --
Train -> 1: 49.80% | 0: 50.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 226ms/step
[AAL][Fold 2] AUC: 0.4360 | F1: 0.3500 | MCC: -0.0724 | ACC: 0.4800
Confusion matrix:
[[51 31]
 [47 21]]


-- Fold 3 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 3s 839ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 241ms/step
[AAL][Fold 3] AUC: 0.5688 | F1: 0.1481 | MCC: 0.1610 | ACC: 0.5400
Confusion matrix:
[[75  1]
 [68  6]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 226ms/step
[AAL][Fold 4] AUC: 0.4788 | F1: 0.2778 | MCC: -0.0840 | ACC: 0.4800
Confusion matrix:
[[57 23]
 [55 15]]


-- Fold 5 --
Train -> 1: 47.50% | 0: 52.50% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 561ms/step
[AAL][Fold 5] AUC: 0.5271 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[81  0]
 [69  0]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 246ms/step
[VRTX][Fold 1] AUC: 0.5470 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 2 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 232ms/step
[VRTX][Fold 2] AUC: 0.4587 | F1: 0.5146 | MCC: -0.1144 | ACC: 0.4467
Confusion matrix:
[[23 51]
 [32 44]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 238ms/step
[VRTX][Fold 3] AUC: 0.4723 | F1: 0.6607 | MCC: 0.0000 | ACC: 0.4933
Confusion matrix:
[[ 0 76]
 [ 0 74]]


-- Fold 4 --
Train -> 1: 51.90% | 0: 48.10% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.40% | 0: 49.60% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 214ms/step
[WDC][Fold 1] AUC: 0.5077 | F1: 0.5375 | MCC: 0.0096 | ACC: 0.5067
Confusion matrix:
[[33 39]
 [35 43]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 229ms/step
[WDC][Fold 2] AUC: 0.5305 | F1: 0.5263 | MCC: 0.0465 | ACC: 0.5200
Confusion matrix:
[[38 42]
 [30 40]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 233ms/step
[WDC][Fold 3] AUC: 0.5204 | F1: 0.5263 | MCC: 0.0490 | ACC: 0.5200
Confusion matrix:
[[38 29]
 [43 40]]


-- Fold 4 --
Train -> 1: 51.20% | 0: 48.80% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.00% | 0: 50.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 237ms/step
[WFC][Fold 1] AUC: 0.4870 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[76  0]
 [74  0]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 238ms/step
[WFC][Fold 2] AUC: 0.4662 | F1: 0.4354 | MCC: -0.1061 | ACC: 0.4467
Confusion matrix:
[[35 44]
 [39 32]]


-- Fold 3 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 56.00% | 0: 44.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 219ms/step
[WFC][Fold 3] AUC: 0.4533 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4400
Confusion matrix:
[[66  0]
 [84  0]]


-- Fold 4 --
Train -> 1: 48.60% | 0: 51.40% (n=1000)
Validation -> 1: 56.00% | 0: 44.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 20,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 226ms/step
[AAL][Fold 1] AUC: 0.4888 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5200
Confusion matrix:
[[78  0]
 [72  0]]


-- Fold 2 --
Train -> 1: 49.80% | 0: 50.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 198ms/step
[AAL][Fold 2] AUC: 0.4476 | F1: 0.1882 | MCC: 0.0124 | ACC: 0.5400
Confusion matrix:
[[73  9]
 [60  8]]


-- Fold 3 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 197ms/step
[AAL][Fold 3] AUC: 0.5103 | F1: 0.0526 | MCC: 0.1178 | ACC: 0.5200
Confusion matrix:
[[76  0]
 [72  2]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 272ms/step
[AAL][Fold 4] AUC: 0.4507 | F1: 0.3721 | MCC: -0.0967 | ACC: 0.4600
Confusion matrix:
[[45 35]
 [46 24]]


-- Fold 5 --
Train -> 1: 47.50% | 0: 52.50% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 218ms/step
[AAL][Fold 5] AUC: 0.5484 | F1: 0.1519 | MCC: 0.0751 | ACC: 0.5533
Confusion matrix:
[[77  4]
 [63  6]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 198ms/step
[VRTX][Fold 1] AUC: 0.5464 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 2 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 317ms/step
[VRTX][Fold 2] AUC: 0.4893 | F1: 0.4394 | MCC: 0.0173 | ACC: 0.5067
Confusion matrix:
[[47 27]
 [47 29]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step
[VRTX][Fold 3] AUC: 0.4860 | F1: 0.6321 | MCC: -0.0531 | ACC: 0.4800
Confusion matrix:
[[ 5 71]
 [ 7 67]]


-- Fold 4 --
Train -> 1: 51.90% | 0: 48.10% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.40% | 0: 49.60% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 235ms/step
[WDC][Fold 1] AUC: 0.4605 | F1: 0.0250 | MCC: -0.0047 | ACC: 0.4800
Confusion matrix:
[[71  1]
 [77  1]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 196ms/step
[WDC][Fold 2] AUC: 0.5277 | F1: 0.5405 | MCC: 0.0963 | ACC: 0.5467
Confusion matrix:
[[42 38]
 [30 40]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 235ms/step
[WDC][Fold 3] AUC: 0.5558 | F1: 0.7162 | MCC: 0.1010 | ACC: 0.5667
Confusion matrix:
[[ 3 64]
 [ 1 82]]


-- Fold 4 --
Train -> 1: 51.20% | 0: 48.80% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.00% | 0: 50.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step
[WFC][Fold 1] AUC: 0.4938 | F1: 0.0000 | MCC: -0.1147 | ACC: 0.4933
Confusion matrix:
[[74  2]
 [74  0]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 219ms/step
[WFC][Fold 2] AUC: 0.5372 | F1: 0.3932 | MCC: 0.0355 | ACC: 0.5267
Confusion matrix:
[[56 23]
 [48 23]]


-- Fold 3 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 56.00% | 0: 44.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 288ms/step
[WFC][Fold 3] AUC: 0.4665 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4400
Confusion matrix:
[[66  0]
 [84  0]]


-- Fold 4 --
Train -> 1: 48.60% | 0: 51.40% (n=1000)
Validation -> 1: 56.00% | 0: 44.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 20,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 322ms/step
[AAL][Fold 1] AUC: 0.6439 | F1: 0.6569 | MCC: 0.1495 | ACC: 0.5333
Confusion matrix:
[[13 65]
 [ 5 67]]


-- Fold 2 --
Train -> 1: 49.80% | 0: 50.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 437ms/step
[AAL][Fold 2] AUC: 0.4690 | F1: 0.2581 | MCC: 0.0240 | ACC: 0.5400
Confusion matrix:
[[69 13]
 [56 12]]


-- Fold 3 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 270ms/step
[AAL][Fold 3] AUC: 0.5466 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[76  0]
 [74  0]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 59.33% | 0: 40.67% (n=150)
Test -> 1: 58.67% | 0: 41.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 305ms/step
[BX][Fold 1] AUC: 0.5081 | F1: 0.5576 | MCC: 0.0224 | ACC: 0.5133
Confusion matrix:
[[31 31]
 [42 46]]


-- Fold 2 --
Train -> 1: 50.30% | 0: 49.70% (n=1000)
Validation -> 1: 58.67% | 0: 41.33% (n=150)
Test -> 1: 56.67% | 0: 43.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 277ms/step
[BX][Fold 2] AUC: 0.3919 | F1: 0.7100 | MCC: -0.0612 | ACC: 0.5533
Confusion matrix:
[[ 1 64]
 [ 3 82]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 56.67% | 0: 43.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 293ms/step
[BX][Fold 3] AUC: 0.4357 | F1: 0.7124 | MCC: 0.0000 | ACC: 0.5533
Confusion matrix:
[[ 0 67]
 [ 0 83]]


-- Fold 4 --
Train -> 1: 53.60% | 0: 46.40% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2

...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 292ms/step
[VRTX][Fold 1] AUC: 0.5464 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 2 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 296ms/step
[VRTX][Fold 2] AUC: 0.5002 | F1: 0.4154 | MCC: -0.0100 | ACC: 0.4933
Confusion matrix:
[[47 27]
 [49 27]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 287ms/step
[VRTX][Fold 3] AUC: 0.5212 | F1: 0.6047 | MCC: 0.1024 | ACC: 0.5467
Confusion matrix:
[[30 46]
 [22 52]]


-- Fold 4 --
Train -> 1: 51.90% | 0: 48.10% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.40% | 0: 49.60% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 273ms/step
[WDC][Fold 1] AUC: 0.5164 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4800
Confusion matrix:
[[72  0]
 [78  0]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step
[WDC][Fold 2] AUC: 0.4677 | F1: 0.4564 | MCC: -0.0767 | ACC: 0.4600
Confusion matrix:
[[35 45]
 [36 34]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 273ms/step
[WDC][Fold 3] AUC: 0.5267 | F1: 0.2453 | MCC: 0.0102 | ACC: 0.4667
Confusion matrix:
[[57 10]
 [70 13]]


-- Fold 4 --
Train -> 1: 51.20% | 0: 48.80% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.00% | 0: 50.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 300ms/step
[WFC][Fold 1] AUC: 0.5373 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[76  0]
 [74  0]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 280ms/step
[WFC][Fold 2] AUC: 0.5099 | F1: 0.5316 | MCC: 0.0222 | ACC: 0.5067
Confusion matrix:
[[34 45]
 [29 42]]


-- Fold 3 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 56.00% | 0: 44.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step
[WFC][Fold 3] AUC: 0.4661 | F1: 0.0000 | MCC: -0.0924 | ACC: 0.4333
Confusion matrix:
[[65  1]
 [84  0]]


-- Fold 4 --
Train -> 1: 48.60% | 0: 51.40% (n=1000)
Validation -> 1: 56.00% | 0: 44.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 20,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 352ms/step
[AAL][Fold 1] AUC: 0.5773 | F1: 0.5109 | MCC: 0.1023 | ACC: 0.5533
Confusion matrix:
[[48 30]
 [37 35]]


-- Fold 2 --
Train -> 1: 49.80% | 0: 50.20% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 348ms/step
[AAL][Fold 2] AUC: 0.4675 | F1: 0.3509 | MCC: -0.0248 | ACC: 0.5067
Confusion matrix:
[[56 26]
 [48 20]]


-- Fold 3 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 343ms/step
[AAL][Fold 3] AUC: 0.4993 | F1: 0.0267 | MCC: 0.0830 | ACC: 0.5133
Confusion matrix:
[[76  0]
 [73  1]]


-- Fold 4 --
Train -> 1: 48.20% | 0: 51.80% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 337ms/step
[AAL][Fold 4] AUC: 0.5218 | F1: 0.1190 | MCC: -0.0704 | ACC: 0.5067
Confusion matrix:
[[71  9]
 [65  5]]


-- Fold 5 --
Train -> 1: 47.50% | 0: 52.50% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 432ms/step
[AAL][Fold 5] AUC: 0.5106 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[81  0]
 [69  0]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 355ms/step
[VRTX][Fold 1] AUC: 0.5445 | F1: 0.6957 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[ 0 70]
 [ 0 80]]


-- Fold 2 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 358ms/step
[VRTX][Fold 2] AUC: 0.4829 | F1: 0.4296 | MCC: -0.0244 | ACC: 0.4867
Confusion matrix:
[[44 30]
 [47 29]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 345ms/step
[VRTX][Fold 3] AUC: 0.4833 | F1: 0.6607 | MCC: 0.0000 | ACC: 0.4933
Confusion matrix:
[[ 0 76]
 [ 0 74]]


-- Fold 4 --
Train -> 1: 51.90% | 0: 48.10% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.40% | 0: 49.60% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 349ms/step
[WDC][Fold 1] AUC: 0.5032 | F1: 0.5829 | MCC: 0.0156 | ACC: 0.5133
Confusion matrix:
[[26 46]
 [27 51]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 529ms/step
[WDC][Fold 2] AUC: 0.5245 | F1: 0.5070 | MCC: 0.0642 | ACC: 0.5333
Confusion matrix:
[[44 36]
 [34 36]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 55.33% | 0: 44.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 339ms/step
[WDC][Fold 3] AUC: 0.5057 | F1: 0.7124 | MCC: 0.0000 | ACC: 0.5533
Confusion matrix:
[[ 0 67]
 [ 0 83]]


-- Fold 4 --
Train -> 1: 51.20% | 0: 48.80% (n=1000)
Validation -> 1: 55.33% | 0: 44.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.00% | 0: 50.00% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 364ms/step
[WFC][Fold 1] AUC: 0.5199 | F1: 0.0920 | MCC: -0.1144 | ACC: 0.4733
Confusion matrix:
[[67  9]
 [70  4]]


-- Fold 2 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 49.33% | 0: 50.67% (n=150)
Test -> 1: 47.33% | 0: 52.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 455ms/step
[WFC][Fold 2] AUC: 0.5300 | F1: 0.5663 | MCC: 0.0563 | ACC: 0.5200
Confusion matrix:
[[31 48]
 [24 47]]


-- Fold 3 --
Train -> 1: 49.30% | 0: 50.70% (n=1000)
Validation -> 1: 47.33% | 0: 52.67% (n=150)
Test -> 1: 56.00% | 0: 44.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 364ms/step
[WFC][Fold 3] AUC: 0.4589 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4400
Confusion matrix:
[[66  0]
 [84  0]]


-- Fold 4 --
Train -> 1: 48.60% | 0: 51.40% (n=1000)
Validation -> 1: 56.00% | 0: 44.00% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━